# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **How many full-time officers were employed in Allegheny county in 2019, and the rate per 1,000 residents?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-07-02 18:02:23 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-07-02T18:02:23.464007")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Retrieval via MCP

The following steps show how data was retrieved through the **U.S. Census Bureau MCP server**. Each step includes reproducible code you can run directly.

---

## Step 1: Get_Police_Employment

**Source:** Fbi Crime Data


**Data retrieved:**
```
{
  "rates": {
    "Law Enforcement Employees per 1,000 People": {
      "2019": 3.22
    }
  },
  "actuals": {
    "Male Officers": {
      "2019": 22937
    },
    "Male Civilians": {
      "2019": 1982
    },
    "Female Officers": {
      "2019": 2728
    },
    "Female Civilians": {
      "2019": 2883
    }
  },
  "tooltips": {
    "Percent of Population Coverage": {
      "Pennsylvania": {
        "2019": 99.82
      }
    }
  },
  "populations": {
    "Participated Population": {
      "2019": 9469659
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "06/2026"
    },
    "last_refresh_date": {
      "UCR": "06/15/2026"
    }
  }
}
```


In [ ]:
# Step 1: Get_Police_Employment

# MCP Tool Call: get_police_employment (via fbi-crime-data)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: get_police_employment
# Arguments:
# {
  "level": "state",
  "from_year": "2019",
  "to_year": "2019",
  "state": "PA"
}

# Result:
result = """{
  "rates": {
    "Law Enforcement Employees per 1,000 People": {
      "2019": 3.22
    }
  },
  "actuals": {
    "Male Officers": {
      "2019": 22937
    },
    "Male Civilians": {
      "2019": 1982
    },
    "Female Officers": {
      "2019": 2728
    },
    "Female Civilians": {
      "2019": 2883
    }
  },
  "tooltips": {
    "Percent of Population Coverage": {
      "Pennsylvania": {
        "2019": 99.82
      }
    }
  },
  "populations": {
    "Participated Population": {
      "2019": 9469659
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "06/2026"
    },
    "last_refresh_date": {
      "UCR": "06/15/2026"
    }
  }
}"""
print(result)


## Step 2: Resolve Geography to FIPS Code

**Source:** Census Data Api

**Geography:** `Allegheny County, Pennsylvania`
**Level:** `County`

**Data retrieved:**
```
Found 1 Matching Geographies:

[
  {
    "id": 9,
    "name": "Philadelphia County, Pennsylvania",
    "summary_level_name": "County",
    "latitude": 40.0094,
    "longitude": -75.1333,
    "for_param": "county:101",
    "in_param": "state:42",
    "weighted_score": 0.5
  }
]
```


In [ ]:
# Step 2: Resolve Geography to FIPS Code

# MCP Tool Call: resolve-geography-fips (via census-data-api)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: resolve-geography-fips
# Arguments:
# {
  "geography_name": "Allegheny County, Pennsylvania",
  "summary_level": "County"
}

# Result:
result = """Found 1 Matching Geographies:

[
  {
    "id": 9,
    "name": "Philadelphia County, Pennsylvania",
    "summary_level_name": "County",
    "latitude": 40.0094,
    "longitude": -75.1333,
    "for_param": "county:101",
    "in_param": "state:42",
    "weighted_score": 0.5
  }
]"""
print(result)


## Step 3: Lookup_Agency

**Source:** Fbi Crime Data


**Data retrieved:**
```
{
  "ALLEGHENY": [
    {
      "ori": "PA0021J00",
      "counties": "ALLEGHENY",
      "is_nibrs": false,
      "latitude": 40.46892,
      "longitude": -79.98092,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Allegheny County Housing Authority",
      "agency_type_name": "Other",
      "nibrs_start_date": null
    },
    {
      "ori": "PA0022800",
      "counties": "ALLEGHENY",
      "is_nibrs": true,
      "latitude": 40.491882,
      "longitude": -79.89781,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Allegheny County Police Department",
      "agency_type_name": "County",
      "nibrs_start_date": "2021-05-01"
    },
    {
      "ori": "PA0022D00",
      "counties": "ALLEGHENY",
      "is_nibrs": false,
    
```


In [ ]:
# Step 3: Lookup_Agency

# MCP Tool Call: lookup_agency (via fbi-crime-data)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: lookup_agency
# Arguments:
# {
  "lookup_type": "by_state",
  "state": "PA",
  "name_filter": "Allegheny County"
}

# Result:
result = """{
  "ALLEGHENY": [
    {
      "ori": "PA0021J00",
      "counties": "ALLEGHENY",
      "is_nibrs": false,
      "latitude": 40.46892,
      "longitude": -79.98092,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Allegheny County Housing Authority",
      "agency_type_name": "Other",
      "nibrs_start_date": null
    },
    {
      "ori": "PA0022800",
      "counties": "ALLEGHENY",
      "is_nibrs": true,
      "latitude": 40.491882,
      "longitude": -79.89781,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Allegheny County Police Department",
      "agency_type_name": "County",
      "nibrs_start_date": "2021-05-01"
    },
    {
      "ori": "PA0022D00",
      "counties": "ALLEGHENY",
      "is_nibrs": false,
      "latitude": 40.441807,
      "longitude": -79.99865,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Allegheny County Port Authority",
      "agency_type_name": "Other",
      "nibrs_start_date": null
    },
    {
      "ori": "PA0024000",
      "counties": "ALLEGHENY",
      "is_nibrs": true,
      "latitude": 40.4432,
      "longitude": -80.147255,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Robinson Township Police Department, Allegheny County",
      "agency_type_name": "City",
      "nibrs_start_date": "2025-01-01"
    },
    {
      "ori": "PA0024100",
      "counties": "ALLEGHENY",
      "is_nibrs": true,
      "latitude": 40.387287,
      "longitude": -80.08466,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Scott Township Police Department, Allegheny County",
      "agency_type_name": "City",
      "nibrs_start_date": "2021-06-01"
    },
    {
      "ori": "PA002A500",
      "counties": "ALLEGHENY",
      "is_nibrs": false,
      "latitude": null,
      "longitude": null,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Alleg"""
print(result)


## Step 4: Resolve Geography to FIPS Code

**Source:** Census Data Api

**Geography:** `Allegheny County, Pennsylvania`

**Data retrieved:**
```
Found 3 Matching Geographies:

[
  {
    "id": 3,
    "name": "Pennsylvania",
    "summary_level_name": "State",
    "latitude": 40.5907,
    "longitude": -77.2098,
    "for_param": "state:42",
    "in_param": "",
    "weighted_score": 1.1333333333333333
  },
  {
    "id": 9,
    "name": "Philadelphia County, Pennsylvania",
    "summary_level_name": "County",
    "latitude": 40.0094,
    "longitude": -75.1333,
    "for_param": "county:101",
    "in_param": "state:42",
    "weighted_score": 1.1
  },
  {
    "id": 12,
    "name": "Philadelphia city, Pennsylvania",
    "summary_level_name": "Place",
    "latitude": 40.0094,
    "longitude": -75.1333,
    "for_param": "place:60000",
    "in_param": "state:42",
    "weighted_score": 0.9488372093023256
  }
]
```


In [ ]:
# Step 4: Resolve Geography to FIPS Code

# MCP Tool Call: resolve-geography-fips (via census-data-api)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: resolve-geography-fips
# Arguments:
# {
  "geography_name": "Allegheny County, Pennsylvania"
}

# Result:
result = """Found 3 Matching Geographies:

[
  {
    "id": 3,
    "name": "Pennsylvania",
    "summary_level_name": "State",
    "latitude": 40.5907,
    "longitude": -77.2098,
    "for_param": "state:42",
    "in_param": "",
    "weighted_score": 1.1333333333333333
  },
  {
    "id": 9,
    "name": "Philadelphia County, Pennsylvania",
    "summary_level_name": "County",
    "latitude": 40.0094,
    "longitude": -75.1333,
    "for_param": "county:101",
    "in_param": "state:42",
    "weighted_score": 1.1
  },
  {
    "id": 12,
    "name": "Philadelphia city, Pennsylvania",
    "summary_level_name": "Place",
    "latitude": 40.0094,
    "longitude": -75.1333,
    "for_param": "place:60000",
    "in_param": "state:42",
    "weighted_score": 0.9488372093023256
  }
]"""
print(result)


## Step 5: Get_Police_Employment

**Source:** Fbi Crime Data


**Data retrieved:**
```
{
  "rates": {
    "Law Enforcement Employees per 1,000 People": {
      "2019": null
    }
  },
  "actuals": {
    "Male Officers": {
      "2019": 203
    },
    "Male Civilians": {
      "2019": 4
    },
    "Female Officers": {
      "2019": 17
    },
    "Female Civilians": {
      "2019": 13
    }
  },
  "tooltips": null,
  "populations": {
    "Participated Population": {
      "2019": 0
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "06/2026"
    },
    "last_refresh_date": {
      "UCR": "06/15/2026"
    }
  }
}
```


In [ ]:
# Step 5: Get_Police_Employment

# MCP Tool Call: get_police_employment (via fbi-crime-data)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: get_police_employment
# Arguments:
# {
  "level": "agency",
  "from_year": "2019",
  "to_year": "2019",
  "state": "PA",
  "ori": "PA0022800"
}

# Result:
result = """{
  "rates": {
    "Law Enforcement Employees per 1,000 People": {
      "2019": null
    }
  },
  "actuals": {
    "Male Officers": {
      "2019": 203
    },
    "Male Civilians": {
      "2019": 4
    },
    "Female Officers": {
      "2019": 17
    },
    "Female Civilians": {
      "2019": 13
    }
  },
  "tooltips": null,
  "populations": {
    "Participated Population": {
      "2019": 0
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "06/2026"
    },
    "last_refresh_date": {
      "UCR": "06/15/2026"
    }
  }
}"""
print(result)


## Step 6: Fetch Aggregate Census Data

**Source:** Census Data Api

**Dataset:** `acs/acs5`
**Year:** 2019
**Variables:** `B01003_001E, NAME`
**Geography (for):** `county:003`
**Geography (in):** `state:42`

**Data retrieved:**
```
Response from acs/acs5:
B01003_001E: 1221744, NAME: Allegheny County, Pennsylvania, state: 42, county: 003
Source: U.S. Census Bureau Data API (https://api.census.gov/data/2019/acs/acs5?get=B01003_001E%2CNAME&for=county%3A003&in=state%3A42&descriptive=false&key=REDACTED)
```


In [ ]:
# Step 6: Fetch Aggregate Census Data

# MCP Tool Call: fetch-aggregate-data (via census-data-api)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: fetch-aggregate-data
# Arguments:
# {
  "dataset": "acs/acs5",
  "year": 2019,
  "get": {
    "variables": [
      "B01003_001E",
      "NAME"
    ]
  },
  "for": "county:003",
  "in": "state:42"
}

# Result:
result = """Response from acs/acs5:
B01003_001E: 1221744, NAME: Allegheny County, Pennsylvania, state: 42, county: 003
Source: U.S. Census Bureau Data API (https://api.census.gov/data/2019/acs/acs5?get=B01003_001E%2CNAME&for=county%3A003&in=state%3A42&descriptive=false&key=REDACTED)"""
print(result)


## Step 7: Get_Police_Employment

**Source:** Fbi Crime Data


**Data retrieved:**
```
{
  "rates": {
    "Law Enforcement Employees per 1,000 People": {
      "2019": null
    }
  },
  "actuals": {
    "Male Officers": {
      "2019": 143
    },
    "Male Civilians": {
      "2019": 13
    },
    "Female Officers": {
      "2019": 21
    },
    "Female Civilians": {
      "2019": 19
    }
  },
  "tooltips": null,
  "populations": {
    "Participated Population": {
      "2019": 0
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "06/2026"
    },
    "last_refresh_date": {
      "UCR": "06/15/2026"
    }
  }
}
```


In [ ]:
# Step 7: Get_Police_Employment

# MCP Tool Call: get_police_employment (via fbi-crime-data)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: get_police_employment
# Arguments:
# {
  "level": "agency",
  "from_year": "2019",
  "to_year": "2019",
  "state": "PA",
  "ori": "PA0020000"
}

# Result:
result = """{
  "rates": {
    "Law Enforcement Employees per 1,000 People": {
      "2019": null
    }
  },
  "actuals": {
    "Male Officers": {
      "2019": 143
    },
    "Male Civilians": {
      "2019": 13
    },
    "Female Officers": {
      "2019": 21
    },
    "Female Civilians": {
      "2019": 19
    }
  },
  "tooltips": null,
  "populations": {
    "Participated Population": {
      "2019": 0
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "06/2026"
    },
    "last_refresh_date": {
      "UCR": "06/15/2026"
    }
  }
}"""
print(result)


## Step 8: Lookup_Agency

**Source:** Fbi Crime Data


**Data retrieved:**
```
{
  "MCKEAN": [
    {
      "ori": "PA0421500",
      "counties": "MCKEAN",
      "is_nibrs": false,
      "latitude": 41.948418,
      "longitude": -78.66883,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Bradford",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "CAMBRIA": [
    {
      "ori": "PA0116500",
      "counties": "CAMBRIA",
      "is_nibrs": false,
      "latitude": 40.270958,
      "longitude": -78.83097,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Johnstown",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "CRAWFORD": [
    {
      "ori": "PA0201900",
   
```


In [ ]:
# Step 8: Lookup_Agency

# MCP Tool Call: lookup_agency (via fbi-crime-data)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: lookup_agency
# Arguments:
# {
  "lookup_type": "by_state",
  "state": "PA",
  "name_filter": "Pittsburgh"
}

# Result:
result = """{
  "MCKEAN": [
    {
      "ori": "PA0421500",
      "counties": "MCKEAN",
      "is_nibrs": false,
      "latitude": 41.948418,
      "longitude": -78.66883,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Bradford",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "CAMBRIA": [
    {
      "ori": "PA0116500",
      "counties": "CAMBRIA",
      "is_nibrs": false,
      "latitude": 40.270958,
      "longitude": -78.83097,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Johnstown",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "CRAWFORD": [
    {
      "ori": "PA0201900",
      "counties": "CRAWFORD",
      "is_nibrs": false,
      "latitude": 41.629498,
      "longitude": -79.66508,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Titusville",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "ALLEGHENY": [
    {
      "ori": "PA0021N00",
      "counties": "ALLEGHENY",
      "is_nibrs": false,
      "latitude": 40.46892,
      "longitude": -79.98092,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Pittsburgh",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    },
    {
      "ori": "PAPPD0000",
      "counties": "ALLEGHENY",
      "is_nibrs": true,
      "latitude": 40.45084,
      "longitude": -80.022484,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Pittsburgh Bureau of Police",
      "agency_type_name": "City",
      "nibrs_start_date": "2023-11-01"
    }
  ],
  "WESTMORELAND": [
    {
      "ori": "PA0655600",
      "counties": "WESTMORELAND",
      "is_nibrs": false,
      "latitude": 40.277306,
      "longitu"""
print(result)


## Step 9: Get_Police_Employment

**Source:** Fbi Crime Data


**Data retrieved:**
```
{
  "rates": {
    "Law Enforcement Employees per 1,000 People": {
      "2019": 3.54
    }
  },
  "actuals": {
    "Male Officers": {
      "2019": 863
    },
    "Male Civilians": {
      "2019": 11
    },
    "Female Officers": {
      "2019": 150
    },
    "Female Civilians": {
      "2019": 40
    }
  },
  "tooltips": null,
  "populations": {
    "Participated Population": {
      "2019": 300548
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "06/2026"
    },
    "last_refresh_date": {
      "UCR": "06/15/2026"
    }
  }
}
```


In [ ]:
# Step 9: Get_Police_Employment

# MCP Tool Call: get_police_employment (via fbi-crime-data)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: get_police_employment
# Arguments:
# {
  "level": "agency",
  "from_year": "2019",
  "to_year": "2019",
  "state": "PA",
  "ori": "PAPPD0000"
}

# Result:
result = """{
  "rates": {
    "Law Enforcement Employees per 1,000 People": {
      "2019": 3.54
    }
  },
  "actuals": {
    "Male Officers": {
      "2019": 863
    },
    "Male Civilians": {
      "2019": 11
    },
    "Female Officers": {
      "2019": 150
    },
    "Female Civilians": {
      "2019": 40
    }
  },
  "tooltips": null,
  "populations": {
    "Participated Population": {
      "2019": 300548
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "06/2026"
    },
    "last_refresh_date": {
      "UCR": "06/15/2026"
    }
  }
}"""
print(result)


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Here's a comprehensive breakdown of law enforcement employment in **Allegheny County in 2019**, based on FBI Crime Data Explorer and U.S. Census Bureau (ACS 5-Year) data:

---

## 🚔 Law Enforcement Employment — Allegheny County, 2019

### County Population (ACS 2019): **1,221,744**

---

### Key Agencies

| Agency | Male Officers | Female Officers | **Total Officers** | Civilians | Rate per 1,000 |
|---|---|---|---|---|---|
| **Pittsburgh Bureau of Police** | 863 | 150 | **1,013** | 51 | **3.54** (city rate) |
| **Allegheny County Police Dept.** | 203 | 17 | **220** | 17 | — |
| **Allegheny County Sheriff's Office** | 143 | 21 | **164** | 32 | — |

---

### Summary

- **Pittsburgh Bureau of Police** is the largest agency in the county with **1,013 sworn officers** at a rate of **3.54 officers per 1,000 city residents** (Pittsburgh population: ~300,548).
- **Allegheny County Police Department** employed **220 sworn officers** serving unincorporated areas of the county.
- **Allegheny County Sheriff's Office** employed **164 sworn officers**.
- Combined, these three major agencies alone account for approximately **1,397 full-time officers** serving the county.

### ⚠️ Important Caveats
- Allegheny County has **130+ municipalities**, many with their own police departments (e.g., Mt. Lebanon, Bethel Park, Penn Hills). A complete county-wide total would require aggregating all municipal agencies.
- The FBI does **not publish a single county-level aggregate** rate — agency-level rates are calculated using each agency's served population.
- The county-wide FBI rate for **Pennsylvania statewide** in 2019 was **3.22 officers per 1,000 residents** (99.82% population coverage), which can serve as a regional benchmark.

**Sources:** FBI Crime Data Explorer (UCR), U.S. Census Bureau ACS 5-Year Estimates (2019)**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-07-02

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-07-02 18:02:23
- **Query**: How many full-time officers were employed in Allegheny county in 2019, and the rate per 1,000 residents?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
